In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

from util.scipy_fisher_exact import fisher_exact
from scipy.stats import ttest_ind, chi2_contingency
import re

from config.config import Config
from config.constants import Constants
C = Constants()

from data_analysis.dataloader.dataloader import LuhaCombinedDataLoader
import librosa

from datetime import datetime
now = lambda: datetime.now().strftime("%Y-%m-%d %H:%M")


/Users/jheitz/miniconda3/envs/phd_new/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataloader = LuhaCombinedDataLoader()
data = dataloader.load_data()

No task specified, loading pictureDescription
No consent_filter specified (consent for further data use), loading all participants
No task specified, loading pictureDescription
No consent_filter specified (consent for further data use), loading all participants
Initializing dataloader Luha 2024 Dataloader (transcript_version google, task pictureDescription) Split full
No task specified, loading pictureDescription
No consent_filter specified (consent for further data use), loading all participants
Initializing dataloader Luha 2026 Dataloader (transcript_version google, task pictureDescription)
Initializing Luha Combined Dataloader (task=pictureDescription, transcript_version=google, split=full, split_2024=full, only_longitudinal_participants=False), waves=None, use_averaged_factor_scores_theory=False
Loading data using Luha Combined Dataloader
Loading data using dataloader Luha 2024 Dataloader
Combining cookieTheft and picnicScene to get pictureDescription... ... done.
Standardizing the

In [3]:
data.longitudinal_waves.shape

(1481,)

In [4]:
# transcript length and audio length for each task


speech_data = {}
for task in ['cookieTheft','picnicScene','journaling']:
    data_task = LuhaCombinedDataLoader(config=Config.from_dict({'config_data': {'task': task}})).load_data()
    speech_data[task] = pd.DataFrame({
        'task': task,
        'sample_name': pd.Series(data_task.sample_names),
        'sample_name_longitudinal': pd.Series(data_task.sample_names_longitudinal),
        'audio_length': pd.Series(data_task.audio_files).apply(lambda path: librosa.get_duration(path=path) if not pd.isna(path) else None),
        'transcript_length': pd.Series(data_task.transcripts).apply(lambda x: len(re.split(r"\s+", x)) if not pd.isna(x) else None),
        'wave': data_task.waves,
        'longitudinal_waves': data_task.longitudinal_waves,
        'year': pd.Series(data_task.dataset_year),
    })
    #speech_data[task]['participant_waves'] =
speech_data_df = pd.concat(speech_data, ignore_index=True)
speech_data_df

No consent_filter specified (consent for further data use), loading all participants
No consent_filter specified (consent for further data use), loading all participants
Initializing dataloader Luha 2024 Dataloader (transcript_version google, task cookieTheft) Split full
No consent_filter specified (consent for further data use), loading all participants
Initializing dataloader Luha 2026 Dataloader (transcript_version google, task cookieTheft)
Initializing Luha Combined Dataloader (task=cookieTheft, transcript_version=google, split=full, split_2024=full, only_longitudinal_participants=False), waves=None, use_averaged_factor_scores_theory=False
Loading data using Luha Combined Dataloader
Loading data using dataloader Luha 2024 Dataloader
Standardizing theory factor scores to zero mean and unit variance. This makes comparison of certain evaluation metrics, beta coefficients etc. more interpretable.


ATTENTION: removing 6 participants with empty transcript on the cookieTheft task (sample

,task,sample_name,sample_name_longitudinal,audio_length,transcript_length,wave,longitudinal_waves,year
0,cookieTheft,41,176874a96f,69.600000,136.0,wave1,[wave1],2024
1,cookieTheft,43,d3109d3bfc,125.220000,239.0,wave1,"[wave1, wave2]",2024
2,cookieTheft,44,ab9278ed99,81.840000,198.0,wave1,"[wave1, wave2]",2024
3,cookieTheft,46,7c95dabae6,65.596375,114.0,wave1,[wave1],2024
4,cookieTheft,49,e76a7e1bce,66.840000,123.0,wave1,[wave1],2024
...,...,...,...,...,...,...,...,...
4413,journaling,1946,10a3893d81,71.520000,193.0,wave3,"[wave2, wave3]",2026
4414,journaling,1947,351ad3a2ff,170.460000,398.0,wave3,"[wave2, wave3]",2026
4415,journaling,1949,f0d5f9966a,104.040000,221.0,wave3,"[wave2, wave3]",2026
4416,journaling,1950,6972fddf91,55.620000,91.0,wave3,"[wave2, wave3]",2026


In [5]:

def clean_audio_length(length):
    if length > 900:  # 15min
        print(f"Outlier length: {length}. Removing...")
        return None
    else:
        return length

speech_data_df['audio_length'] = speech_data_df['audio_length'].apply(clean_audio_length)

Outlier length: 3587.4. Removing...
Outlier length: 1403.52. Removing...


In [6]:
speech_data_mean = speech_data_df.groupby("sample_name").agg({
    'audio_length': 'mean',
    'transcript_length': 'mean'
}).reset_index().rename(columns={'audio_length': 'mean_audio_length', 'transcript_length': 'mean_transcript_length'})
speech_data_mean


,sample_name,mean_audio_length,mean_transcript_length
0,1000,231.240000,506.333333
1,1001,66.940000,131.666667
2,1002,91.740000,214.666667
3,1003,70.220000,101.666667
4,1004,90.660000,161.000000
...,...,...,...
1476,993,49.440000,135.333333
1477,994,85.000000,224.666667
1478,997,94.772444,242.000000
1479,998,71.640000,169.000000


In [7]:
speech_data_df.longitudinal_waves.iloc[0]

['wave1']

In [8]:

speech_data_subsets = {
    "full": speech_data_df,
    'full_wave1': speech_data_df[(speech_data_df.wave == "wave1")],
    'full_wave2': speech_data_df[(speech_data_df.wave == "wave2")],
    'full_wave3': speech_data_df[(speech_data_df.wave == "wave3")],
    'wave1_only': speech_data_df[speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave1'])],
    'cohort_a': speech_data_df[speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave1', 'wave2'])],
    'cohortA_wave1': speech_data_df[(speech_data_df.wave == "wave1") & (speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave1', 'wave2']))],
    'cohortA_wave2': speech_data_df[(speech_data_df.wave == "wave2") & (speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave1', 'wave2']))],
    'wave2_only': speech_data_df[speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave2'])],
    'wave2_new_participants': speech_data_df[speech_data_df.longitudinal_waves.apply(lambda waves: waves in [['wave2'], ['wave2', 'wave3']])],
    'cohortB': speech_data_df[speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave2', 'wave3'])],
    'cohortB_wave2': speech_data_df[(speech_data_df.wave == "wave2") & (speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave2', 'wave3']))],
    'cohortB_wave3': speech_data_df[(speech_data_df.wave == "wave3") & (speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave2', 'wave3']))],
    'wave3_only': speech_data_df[speech_data_df.longitudinal_waves.apply(lambda waves: waves == ['wave3'])],
}

task_renaming = {
    'cookieTheft': 'Cookie Theft',
    'picnicScene': 'Picnic Scene',
    'journaling': 'Journaling',
}


def get_speech_task_stats(subset) -> pd.DataFrame:
    speech_data_per_task = speech_data_subsets[subset].groupby(['task']).agg({
        'audio_length': 'mean',
        'transcript_length': 'mean'
    }).rename(columns={'audio_length': 'Mean Audio Length (s)', 'transcript_length': 'Mean Transcript Length (\\# words)'}).T
    speech_data_per_task = speech_data_per_task.rename(columns=task_renaming)
    speech_data_per_task.columns.name = None
    return speech_data_per_task

speech_data_per_task_df = get_speech_task_stats("full")
#display(speech_data_per_task_df)
speech_data_per_task_latex = speech_data_per_task_df.to_latex(column_format='l' + 'c' * (speech_data_per_task_df.shape[1]),  float_format='%.2f',
                                                           caption="Mean audio and transcript lengths for each spontaneous speech task across all participants and waves",
                                                           label="tab:speech_stats_per_task")
print(speech_data_per_task_latex)


\begin{table}
\caption{Mean audio and transcript lengths for each spontaneous speech task across all participants and waves}
\label{tab:speech_stats_per_task}
\begin{tabular}{lccc}
\toprule
 & Cookie Theft & Journaling & Picnic Scene \\
\midrule
Mean Audio Length (s) & 70.15 & 92.96 & 86.62 \\
Mean Transcript Length (\# words) & 149.33 & 190.94 & 189.36 \\
\bottomrule
\end{tabular}
\end{table}



In [9]:
targets = ['composite_memory', 'composite_language', 'composite_executive_function', 'composite_speed']

df = data.demographics.copy()
df['mean_cognition'] = data.mean_composite_cognitive_score
df['sample_name'] = data.sample_names
df['sample_name_longitudinal'] = data.sample_names_longitudinal
df['wave'] = data.waves
df['longitudinal_waves'] = data.longitudinal_waves
df['year'] = data.dataset_year
df = df.merge(speech_data_mean, on='sample_name', how='left')


#df = df.dropna(subset=['mean_cognition'])
for composite_score in data.factor_scores_theory.columns:
    df[composite_score] = data.factor_scores_theory[composite_score]



df

,sex_prolific,ethnicity,country_of_birth,country_of_residence,nationality,age_prolific,age,gender,education,language,...,sample_name_longitudinal,wave,longitudinal_waves,year,mean_audio_length,mean_transcript_length,composite_memory,composite_language,composite_speed,composite_executive_function
0,Female,White,Malta,United Kingdom,United Kingdom,61.0,61.0,f,bachelor,english_british,...,176874a96f,wave1,[wave1],2024,67.240000,134.666667,NaN,NaN,NaN,NaN
1,Female,White,United Kingdom,United Kingdom,United Kingdom,63.0,63.0,f,master,english_british,...,d3109d3bfc,wave1,"[wave1, wave2]",2024,111.080000,201.333333,NaN,NaN,NaN,NaN
2,Male,White,United Kingdom,United Kingdom,United Kingdom,64.0,64.0,m,bachelor,english_british,...,ab9278ed99,wave1,"[wave1, wave2]",2024,74.320000,178.000000,NaN,NaN,NaN,NaN
3,Male,White,United Kingdom,United Kingdom,United Kingdom,62.0,62.0,m,high_school,english_british,...,7c95dabae6,wave1,[wave1],2024,70.012042,127.333333,NaN,NaN,NaN,NaN
4,Male,White,United Kingdom,United Kingdom,United Kingdom,73.0,73.0,m,bachelor,english_british,...,e76a7e1bce,wave1,[wave1],2024,72.040000,136.000000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,Female,White,United Kingdom,United Kingdom,United Kingdom,61.0,61.0,f,bachelor,english_british,...,10a3893d81,wave3,"[wave2, wave3]",2026,71.080000,197.333333,0.009962,1.059569,0.286316,0.523724
1477,Female,White,United Kingdom,United Kingdom,United Kingdom,63.0,62.0,f,high_school,english_british,...,351ad3a2ff,wave3,"[wave2, wave3]",2026,121.800000,262.000000,0.433639,1.380432,0.389651,1.147031
1478,Male,White,United Kingdom,United Kingdom,United Kingdom,70.0,70.0,m,bachelor,english_british,...,f0d5f9966a,wave3,"[wave2, wave3]",2026,89.560000,181.333333,-0.545741,-0.021867,0.192694,0.232663
1479,Male,White,United Kingdom,United Kingdom,United Kingdom,61.0,61.0,m,bachelor,english_british,...,6972fddf91,wave3,"[wave2, wave3]",2026,69.840000,139.666667,-0.149979,0.536164,0.849361,0.768990


In [10]:

subsets = {
    "full": df,
    'full_wave1': df[(df.wave == "wave1")],
    'full_wave2': df[(df.wave == "wave2")],
    'full_wave3': df[(df.wave == "wave3")],
    'wave1_only': df[df.longitudinal_waves.apply(lambda waves: waves == ['wave1'])],
    'cohortA': df[df.longitudinal_waves.apply(lambda waves: waves == ['wave1', 'wave2'])],
    'cohortA_wave1': df[(df.wave == "wave1") & (df.longitudinal_waves.apply(lambda waves: waves == ['wave1', 'wave2']))],
    'cohortA_wave2': df[(df.wave == "wave2") & (df.longitudinal_waves.apply(lambda waves: waves == ['wave1', 'wave2']))],
    'wave2_only': df[df.longitudinal_waves.apply(lambda waves: waves == ['wave2'])],
    'wave2_new_participants': df[(df.wave == "wave2") & (df.longitudinal_waves.apply(lambda waves: waves in [['wave2'], ['wave2', 'wave3']]))],
    'cohortB': df[df.longitudinal_waves.apply(lambda waves: waves == ['wave2', 'wave3'])],
    'cohortB_wave2': df[(df.wave == "wave2") & (df.longitudinal_waves.apply(lambda waves: waves == ['wave2', 'wave3']))],
    'cohortB_wave3': df[(df.wave == "wave3") & (df.longitudinal_waves.apply(lambda waves: waves == ['wave2', 'wave3']))],
    'wave3_only': df[df.longitudinal_waves.apply(lambda waves: waves == ['wave3'])],

}



demographic_statistics_text = defaultdict(dict)
demographic_statistics = defaultdict(dict)

def statistics_categorical(col):
     for split, subset in subsets.items():
        value_counts = subset[col].value_counts()
        values_counts_with_percentage = {val: f"{count} ({count/subset[col].shape[0]*100:.1f}%)" for val, count in value_counts.items()}

        if col == 'education':
            value_counts_here = subset['education_binary'].value_counts()
            #values_counts_with_percentage_addition = {f"(Summary) {val}": f"{count} ({count/subset[col].shape[0]*100:.1f}%)" for val, count in value_counts_here.items()}
            values_counts_with_percentage_addition = {f"{val}": f"{count} ({count/subset[col].shape[0]*100:.1f}%)" for val, count in value_counts_here.items()}
            values_counts_with_percentage = {**values_counts_with_percentage, **values_counts_with_percentage_addition}

        demographic_statistics_text[col][split] = values_counts_with_percentage
        demographic_statistics[col][split] = value_counts


def statistics_continuous(col):
    for split, subset in subsets.items():
        mean, std = np.mean(subset[col]), np.std(subset[col])
        mean_std_str = f"{mean:.2f} +- {std:.2f}"

        demographic_statistics_text[col][split] = mean_std_str
        demographic_statistics[col][split] = subset[col]




for col in ['age']:
    statistics_continuous(col)

for col in ['gender_unified', 'country', 'language', 'education', 'ethnicity', ]:
   statistics_categorical(col)

for col in ['socioeconomic', 'mean_transcript_length', 'mean_audio_length', 'mean_cognition', 'composite_language', 'composite_executive_function', 'composite_memory', 'composite_speed']:
    statistics_continuous(col)


In [11]:



col_renaming = {
    'age': 'Age',
    'gender_unified': 'Gender',
    'country': 'Country',
    'language': 'Language',
    'education': 'Education',
    'ethnicity': 'Ethnicity',
    'socioeconomic': 'Socioeconomic Status',
    'mean_transcript_length': 'Mean Transcript Length (\\# words)',
    'mean_audio_length': 'Mean Audio Length (s)',
    'mean_cognition': 'Mean Cognitive Score',
    'composite_language': 'Language',
    'composite_executive_function': 'Executive Function',
    'composite_memory': 'Memory',
    'composite_speed': 'Speed',
}
value_renaming = {
    'f': 'Female',
    'm': 'Male',
    'usa': 'USA',
    'uk': 'UK',
    'bachelor': 'Bachelor',
    'master': 'Master',
    'high_school': 'High school',
    'vocational': 'Vocational',
    'phd': 'PhD',
    'less_than_highschool': 'Less than high school',
    'high-education': 'High Education',
    'low-education': 'Low Education',
    '(Summary) high-education': '(Summary) High Education',
    '(Summary) low-education': '(Summary) Low Education',

}
table_column_renaming = {
    'wave1_only': '\\makecell[lc]{Wave 1-only\\\\participants}',
    'cohortA_wave1': '\\makecell[lc]{Cohort A\\\\(Wave 1)}',
    'cohortA_wave2': '\\makecell[lc]{Cohort A\\\\(Wave 2)}',
    'full_wave1': 'Wave 1',
    'full_wave2': 'Wave 2',
    'full_wave3': 'Wave 3',
    'wave2_only': 'Non-returners',
    'wave2_new_participants': '\\makecell[lc]{Wave 2 - New\\\\participants}',
    'cohortB_wave2': '\\makecell[lc]{Cohort B\\\\(Baseline)}',
    'cohortB_wave3': '\\makecell[lc]{Cohort B\\\\(Follow-up)}',
    'pval': '$p$-value',
    'group': '',
}

def format_pval(pval):
    if pval < 0.01:
        return "$<0.01$"
    else:
        return f"${pval:.2f}$"

def demographic_table_two_subset_with_stats(subset1, subset2, ignore_cols=[], ignore_values=[]):
    table_rows = []
    first_row = {'group': '\\# Participants', subset1: subsets[subset1].shape[0], subset2: subsets[subset2].shape[0], 'pval': ''}
    table_rows.append(first_row)
    for col in demographic_statistics_text.keys():
        if col in ignore_cols:
            continue
        if isinstance(demographic_statistics_text[col][subset1], dict):
            values = list(demographic_statistics[col][subset1].keys())
            values += [c for c in demographic_statistics[col][subset2].keys() if c not in values]
            contingency_table = pd.DataFrame([
                {val: demographic_statistics[col][subset1].get(val, 0) for val in values},
                {val: demographic_statistics[col][subset2].get(val, 0) for val in values},
            ])
            fisher_exact_pval = fisher_exact(contingency_table.T).pvalue
            expected_frequencies = contingency_table.iloc[1,:] / np.sum(contingency_table.iloc[1,:]) * np.sum(contingency_table.iloc[0,:])
            #display(contingency_table)
            chisquare_pval = chi2_contingency(contingency_table).pvalue
            values_text = list(demographic_statistics_text[col][subset1].keys())
            values_text += [c for c in demographic_statistics_text[col][subset2].keys() if c not in values_text]
            for i, val in enumerate(values_text):
                if i == 0:
                    table_rows.append({'group': col_renaming.get(col, col), subset1: '', subset2: '', 'pval': format_pval(fisher_exact_pval)})  #  / {chisquare_pval}
                if val in ignore_values:
                    continue
                row = {'group': f"\\quad {value_renaming.get(val, val)}", subset1: demographic_statistics_text[col][subset1].get(val, f"0 (0%)"), subset2: demographic_statistics_text[col][subset2].get(val, f"0 (0%)"), 'pval': ''}
                table_rows.append(row)

        else:
            ttest = ttest_ind(demographic_statistics[col][subset1], demographic_statistics[col][subset2], nan_policy='omit')
            #print(f"   col: {list(demographic_statistics[col][group1'])}", "|", list(demographic_statistics[col][group2]), ttest.pvalue)
            row = {'group': col_renaming.get(col, col), subset1: demographic_statistics_text[col][subset1], subset2: demographic_statistics_text[col][subset2], "pval": format_pval(ttest.pvalue)}
            table_rows.append(row)

    table_df = pd.DataFrame(table_rows)
    table_df.columns.name = None
    table_df.index.name = None,
    return table_df



def demographic_table_n_subsets(subsets_list, ignore_cols=[], ignore_values=[]):
    """
    Build a demographic table comparing N subsets.

    Args:
        subsets_list: list of subset names (e.g., ['wave1_only', 'cohortA_wave1', 'cohortB_wave2'])
    """
    table_rows = []

    # First row: number of participants per subset
    first_row = {'group': '\\# Participants'}
    for s in subsets_list:
        first_row[s] = subsets[s].shape[0]
    table_rows.append(first_row)

    for col in demographic_statistics_text.keys():
        if col in ignore_cols:
            continue

        # Categorical variable (dict of value -> count/text)
        if isinstance(demographic_statistics_text[col][subsets_list[0]], dict):
            # Collect union of all values across all subsets, preserving order
            values_text = []
            for s in subsets_list:
                for v in demographic_statistics_text[col][s].keys():
                    if v not in values_text:
                        values_text.append(v)

            for i, val in enumerate(values_text):
                if i == 0:
                    # Header row for this categorical variable
                    header_row = {'group': col_renaming.get(col, col)}
                    for s in subsets_list:
                        header_row[s] = ''
                    table_rows.append(header_row)

                if val in ignore_values:
                    continue

                row = {'group': f"\\quad {value_renaming.get(val, val)}"}
                for s in subsets_list:
                    row[s] = demographic_statistics_text[col][s].get(val, "0 (0%)")
                table_rows.append(row)

        # Continuous variable
        else:
            row = {'group': col_renaming.get(col, col)}
            for s in subsets_list:
                row[s] = demographic_statistics_text[col][s]
            table_rows.append(row)

    table_df = pd.DataFrame(table_rows)
    table_df.columns.name = None
    table_df.index.name = None
    return table_df


def get_sig_stars(pval):
    if pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < 0.05:
        return '*'
    else:
        return ''


def demographic_table_n_subsets_with_tests(subsets_list, comparison_pairs=None,
                                 ignore_cols=[], ignore_values=[]):
    """
    Args:
        subsets_list: list of subset names (column order in the table)
        comparison_pairs: list of (subset_a, subset_b) tuples. For each pair,
            a separate '<sb>_sig' column is added immediately after subset_b
            containing significance stars.
    """
    table_rows = []
    comparison_pairs = comparison_pairs or []

    # Map from subset_b -> sig column name, for quick lookup
    sig_col_for = {sb: f"{sb}_sig" for _, sb in comparison_pairs}

    # Build the full ordered column list: group, then each subset, with sig
    # column inserted after any subset that is the 'b' of a pair.
    ordered_cols = ['group']
    for s in subsets_list:
        ordered_cols.append(s)
        if s in sig_col_for:
            ordered_cols.append(sig_col_for[s])

    def empty_row():
        return {c: '' for c in ordered_cols}

    # First row: number of participants
    first_row = empty_row()
    first_row['group'] = '\\# Participants'
    for s in subsets_list:
        first_row[s] = subsets[s].shape[0]
    table_rows.append(first_row)

    for col in demographic_statistics_text.keys():
        if col in ignore_cols:
            continue

        if isinstance(demographic_statistics_text[col][subsets_list[0]], dict):
            # Categorical
            values_text = []
            for s in subsets_list:
                for v in demographic_statistics_text[col][s].keys():
                    if v not in values_text:
                        values_text.append(v)

            # Compute Fisher's exact p-values per pair
            pair_stars = {}  # subset_b -> stars
            for sa, sb in comparison_pairs:
                values = list(demographic_statistics[col][sa].keys())
                values += [c for c in demographic_statistics[col][sb].keys() if c not in values]
                contingency = pd.DataFrame([
                    {v: demographic_statistics[col][sa].get(v, 0) for v in values},
                    {v: demographic_statistics[col][sb].get(v, 0) for v in values},
                ])
                pval = fisher_exact(contingency.T).pvalue
                pair_stars[sb] = get_sig_stars(pval)

            for i, val in enumerate(values_text):
                if i == 0:
                    header_row = empty_row()
                    header_row['group'] = col_renaming.get(col, col)
                    # Stars belong on the header row, in the sig column for each pair
                    for sb, stars in pair_stars.items():
                        header_row[sig_col_for[sb]] = stars
                    table_rows.append(header_row)

                if val in ignore_values:
                    continue

                row = empty_row()
                row['group'] = f"\\quad {value_renaming.get(val, val)}"
                for s in subsets_list:
                    row[s] = demographic_statistics_text[col][s].get(val, "0 (0%)")
                table_rows.append(row)

        else:
            # Continuous
            pair_stars = {}
            for sa, sb in comparison_pairs:
                ttest = ttest_ind(demographic_statistics[col][sa],
                                  demographic_statistics[col][sb],
                                  nan_policy='omit')
                pair_stars[sb] = get_sig_stars(ttest.pvalue)

            row = empty_row()
            row['group'] = col_renaming.get(col, col)
            for s in subsets_list:
                row[s] = demographic_statistics_text[col][s]
            for sb, stars in pair_stars.items():
                row[sig_col_for[sb]] = stars
            table_rows.append(row)

    table_df = pd.DataFrame(table_rows, columns=ordered_cols)
    table_df.columns.name = None
    table_df.index.name = None
    return table_df


In [12]:
# Attrition analysis Cohort A


subset1 = 'wave1_only'
subset2 = 'cohortA_wave1'

ignore_cols = ['language', 'ethnicity']
ignore_values = ['DATA_EXPIRED', 'no_answer', '(Summary) no_answer', 'bachelor', 'master', 'vocational', 'less_than_highschool', 'phd', 'high_school']


table_df = demographic_table_two_subset_with_stats(subset1, subset2, ignore_cols=ignore_cols, ignore_values=ignore_values)
table_df

,group,wave1_only,cohortA_wave1,pval
"(None,)",,,,
0,\# Participants,698,305,
1,Age,65.49 +- 4.84,65.42 +- 4.72,$0.83$
2,Gender,,,$0.12$
3,\quad Female,439 (62.9%),176 (57.7%),
4,\quad Male,259 (37.1%),129 (42.3%),
5,Country,,,$<0.01$
6,\quad USA,381 (54.6%),120 (39.3%),
7,\quad UK,316 (45.3%),185 (60.7%),
8,Education,,,$0.29$


In [14]:
table_latex = table_df.rename(columns=table_column_renaming).to_latex(index=False, column_format='l' + 'l' * (len(table_df.columns) - 1),
                                                                      caption="Baseline demographic and cognitive characteristics of the original $n=1003$ participants who returned for a follow-up assessment ($n=305$, Cohort A) and those who did not ($n=698$).",
                                                                      label="tab:attrition_cohortA").replace("%", "\\%").replace("_", "-").replace("+-", "$\\pm$")
print(table_latex)

\begin{table}
\caption{Baseline demographic and cognitive characteristics of the original $n=1003$ participants who returned for a follow-up assessment ($n=305$, Cohort A) and those who did not ($n=698$).}
\label{tab:attrition-cohortA}
\begin{tabular}{llll}
\toprule
 & \makecell[lc]{Wave 1-only\\participants} & \makecell[lc]{Cohort A\\(Wave 1)} & $p$-value \\
\midrule
\# Participants & 698 & 305 &  \\
Age & 65.49 $\pm$ 4.84 & 65.42 $\pm$ 4.72 & $0.83$ \\
Gender &  &  & $0.12$ \\
\quad Female & 439 (62.9\%) & 176 (57.7\%) &  \\
\quad Male & 259 (37.1\%) & 129 (42.3\%) &  \\
Country &  &  & $<0.01$ \\
\quad USA & 381 (54.6\%) & 120 (39.3\%) &  \\
\quad UK & 316 (45.3\%) & 185 (60.7\%) &  \\
Education &  &  & $0.29$ \\
\quad High Education & 431 (61.7\%) & 170 (55.7\%) &  \\
\quad Low Education & 264 (37.8\%) & 134 (43.9\%) &  \\
Socioeconomic Status & 5.57 $\pm$ 1.67 & 5.60 $\pm$ 1.55 & $0.78$ \\
Mean Transcript Length (\# words) & 172.61 $\pm$ 126.56 & 175.96 $\pm$ 69.30 & $0.66$ \\
Mea

In [15]:
# Attrition analysis Cohort B


subset1 = 'wave2_only'
subset2 = 'cohortB_wave2'

ignore_cols = ['language', 'ethnicity']
ignore_values = ['DATA_EXPIRED', 'no_answer', '(Summary) no_answer', 'bachelor', 'master', 'vocational', 'less_than_highschool', 'phd', 'high_school']


table_df = demographic_table_two_subset_with_stats(subset1, subset2, ignore_cols=ignore_cols, ignore_values=ignore_values)
table_df

,group,wave2_only,cohortB_wave2,pval
"(None,)",,,,
0,\# Participants,33,70,
1,Age,66.24 +- 5.06,65.34 +- 5.94,$0.46$
2,Gender,,,$0.67$
3,\quad Female,19 (57.6%),44 (62.9%),
4,\quad Male,14 (42.4%),26 (37.1%),
5,Country,,,$1.00$
6,\quad UK,33 (100.0%),70 (100.0%),
7,Education,,,$0.19$
8,\quad High Education,24 (72.7%),47 (67.1%),


In [16]:
table_latex = table_df.rename(columns=table_column_renaming).to_latex(index=False, column_format='l' + 'l' * (len(table_df.columns) - 1),
                                                                      caption="Attrition analysis Cohort B: Baseline demographic and cognitive characteristics of the $n=103$ participants newly recruited for replication. We compare the participants who returned for a follow-up assessment ($n=70$, Cohort B) and those who did not ($n=33$).",
                                                                      label="tab:attrition_cohortB").replace("%", "\\%").replace("_", "-").replace("+-", "$\\pm$")
print(table_latex)

\begin{table}
\caption{Attrition analysis Cohort B: Baseline demographic and cognitive characteristics of the $n=103$ participants newly recruited for replication. We compare the participants who returned for a follow-up assessment ($n=70$, Cohort B) and those who did not ($n=33$).}
\label{tab:attrition-cohortB}
\begin{tabular}{llll}
\toprule
 & Non-returners & \makecell[lc]{Cohort B\\(Baseline)} & $p$-value \\
\midrule
\# Participants & 33 & 70 &  \\
Age & 66.24 $\pm$ 5.06 & 65.34 $\pm$ 5.94 & $0.46$ \\
Gender &  &  & $0.67$ \\
\quad Female & 19 (57.6\%) & 44 (62.9\%) &  \\
\quad Male & 14 (42.4\%) & 26 (37.1\%) &  \\
Country &  &  & $1.00$ \\
\quad UK & 33 (100.0\%) & 70 (100.0\%) &  \\
Education &  &  & $0.19$ \\
\quad High Education & 24 (72.7\%) & 47 (67.1\%) &  \\
\quad Low Education & 9 (27.3\%) & 23 (32.9\%) &  \\
Socioeconomic Status & 5.88 $\pm$ 1.54 & 5.73 $\pm$ 1.51 & $0.66$ \\
Mean Transcript Length (\# words) & 163.42 $\pm$ 57.08 & 187.06 $\pm$ 75.06 & $0.12$ \\
Mean Audi